In [57]:
%reload_ext autoreload
%autoreload 1
%aimport my_graphrag
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from my_graphrag import KBGraphRAG
from openai import AsyncOpenAI
load_dotenv(".env", override=True)

True

In [58]:
client = AsyncOpenAI(
    base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
    api_key="ollama",
)
# Separate client for real OpenAI-hosted models (e.g. gpt-5-mini used by DriftSearch's
# reasoning/response calls) — `client` above points at local Ollama and doesn't serve them.
driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    notifications_min_severity="OFF",
)
driver.verify_connectivity()

# Routes to a local Ollama server (default http://localhost:11434/v1, override with OLLAMA_BASE_URL).
# Pull the model first: `ollama pull llama3.2`
# gpt-oss:20b took ~8h for 86 chunks at max_workers=10 (349s/chunk, fully serialized —
# Ollama queues requests per-model by default, so max_workers had no effect). Switched to
# llama3.2 for much faster local inference and dropped max_workers to 2 to avoid
# overloading the single Ollama instance.
kb = KBGraphRAG(driver=driver, max_workers=2, model="ollama/llama3.2")
print(kb.model)

ollama/llama3.2


In [ ]:
#Define example texts and entity types
example_texts = [
    "Tom is an American",
    "Tom lives in Washington DC", 
    "Tom went to school in Utah"
],
allowed_entities = ["Person", "Nationality", "Location"]
#Extract entities and relationships
result = kb.extract_nodes_and_rels(example_texts, allowed_entities)
print(result)
kb.query("MATCH (e:__Entity__) RETURN e.name, e.description LIMIT 20")

In [7]:
await kb.summarize_nodes_and_rels()

Summarizing nodes: 0it [00:00, ?it/s]
Summarizing relationships: 0it [00:00, ?it/s]


'Successfuly summarized nodes and relationships'

In [8]:
await kb.summarize_communities()

Leiden algorithm identified 1 community levels with 1 communities on the last level.


Summarizing communities: 100%|██████████| 1/1 [00:02<00:00,  2.99s/it]


'Generated 1 community summaries'

In [59]:
from getpass import getpass
from typing import List
import requests
import tiktoken
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.workflow import (
    Event,
    Context,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)
import json_repair

In [60]:
book = requests.get('https://www.gutenberg.org/cache/epub/11/pg11.txt').text   
splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=20)
class EntitySummarization(Event):
    pass

class CommunitySummarization(Event):
    pass

class CommunityEmbeddings(Event):
    pass

class EntityEmbeddings(Event):
    pass

In [61]:
MIN_COMMUNITY_RATING = 3
# Local Ollama embedding model — pull it first: `ollama pull nomic-embed-text`
TEXT_EMBEDDING_MODEL = "nomic-embed-text"

class MSGraphRAGIngestion(Workflow):
    @step
    async def entity_extraction(self, ev: StartEvent) -> EntitySummarization:
        chunks = splitter.split_text(ev.text)
        await kb.extract_nodes_and_rels(chunks, ev.allowed_entities)
        return EntitySummarization()

    @step
    async def entity_summarization(
        self, ev: EntitySummarization
    ) -> CommunitySummarization:
        await kb.summarize_nodes_and_rels()
        return CommunitySummarization()

    @step
    async def community_summarization(
        self, ev: CommunitySummarization
    ) -> CommunityEmbeddings:
        await kb.summarize_communities()
        return CommunityEmbeddings()

    @step
    async def community_embeddings(self, ev: CommunityEmbeddings) -> EntityEmbeddings:
        communities = kb.query(
            """
            MATCH (c:__Community__)
            WHERE c.summary IS NOT NULL AND c.rating > $min_community_rating
            RETURN coalesce(c.title, "") + " " + c.summary AS community_description, c.id AS community_id
            """,
            params={"min_community_rating": MIN_COMMUNITY_RATING},
        )
        if communities:
            response = await client.embeddings.create(
                input=[c["community_description"] for c in communities],
                model=TEXT_EMBEDDING_MODEL,
            )
            embeds = []
            for community, embedding in zip(communities, response.data):
                embeds.append(
                    {
                        "community_id": community["community_id"],
                        "embedding": embedding.embedding,
                    }
                )
            kb.query(
                """UNWIND $data as row
                MATCH (c:__Community__ {id: row.community_id})
                CALL db.create.setNodeVectorProperty(c, 'embedding', row.embedding)""",
                params={"data": embeds},
            )
            kb.query(
                "CREATE VECTOR INDEX community IF NOT EXISTS FOR (c:__Community__) ON c.embedding"
            )
        else:
            print("No community was summarized")
        return EntityEmbeddings()

    @step
    async def entity_embeddings(self, ev: EntityEmbeddings) -> StopEvent:
        entities = kb.query("""
                MATCH (e:__Entity__)
                WHERE e.summary IS NOT NULL
                RETURN coalesce(e.name, "") + " " + e.summary AS entity_description, e.name AS entity_name
                """)
        if entities:
            response = await client.embeddings.create(
                input=[e["entity_description"] for e in entities],
                model=TEXT_EMBEDDING_MODEL,
            )
            embeds = []
            for entity, embedding in zip(entities, response.data):
                embeds.append(
                    {
                        "entity_name": entity["entity_name"],
                        "embedding": embedding.embedding,
                    }
                )
            kb.query(
                """UNWIND $data as row
                    MATCH (e:__Entity__ {name: row.entity_name})
                    CALL db.create.setNodeVectorProperty(e, 'embedding', row.embedding)""",
                params={"data": embeds},
            )
            kb.query(
                "CREATE VECTOR INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON e.embedding"
            )
        else:
            print("No entity was summarized")
        return StopEvent()

In [23]:
w = MSGraphRAGIngestion(timeout=36000, verbose=False)
result = await w.run(text=book, allowed_entities = ["PERSON", "ORGANIZATION", "LOCATION", "EVENT", "ARTIFACT"])     

Summarizing relationships: 100%|██████████| 109/109 [03:35<00:00,  1.98s/it]


Leiden algorithm identified 3 community levels with 25 communities on the last level.


Summarizing communities:  24%|██▍       | 7/29 [03:49<13:10, 35.91s/it]

JSON parsing error for community summary: Community summary JSON missing expected scalar fields: [['Data: Entities (1)'], ['Data: Entities (2)', 'Relationships (1)'], ['Data: Entities (5)'], ['Data: Entities (1-5)', 'Relationships (1-5)'], ['Data: Entities (1)'], ['Data: Entities (1-5)', 'Relationships (1-5)'], {'title': 'Father William and Family', 'summary': 'The community revolves around Father William and his family, with a focus on relationships between characters and potential insights into family dynamics and human behavior.', 'rating': 4.0, 'rating_explanation': 'The impact severity rating is moderate due to the exploration of family dynamics and relationships between characters.', 'findings': [{'summary': "Father William's Character Development", 'explanation': 'Father William is a recurring character in stories, a father figure, and the protagonist of a poem and story, often depicted as a wise mentor. This highlight the significance of character development in this community.

Summarizing communities:  28%|██▊       | 8/29 [04:28<12:57, 37.01s/it]

...

Summarizing communities:  72%|███████▏  | 21/29 [38:07<26:44, 200.52s/it]  

JSON parsing error for community summary: Community summary JSON missing expected scalar fields: [{'title': 'Aurelia and Quintara', 'summary': 'Aurelia and Quintara are two entities that are interlinked through various descriptions and relationships. Aurelia is a country with a history of holding hostages, while Quintara is a country located in the Aurelia region. These descriptions suggest that there is a relationship between the two countries, potentially involving the release of hostages. However, more information is needed to determine the nature of this relationship.', 'rating': 3.5, 'rating_explanation': 'The impact severity rating is moderate due to the potential for instability or conflict between the two countries.'}, {'summary': 'Verdant Oasis Plaza is the central location in this community, serving as the location for the Unity March. This plaza is the common link between all other entities, suggesting its significance in the community.', 'explanation': 'Verdant Oasis Plaza 

Summarizing communities:  86%|████████▌ | 25/29 [41:00<05:30, 82.67s/it] 

JSON parsing error for community summary: Community summary JSON missing expected scalar fields: [["start_entity['id'"], ['id'], ['id'], [0, 5], ['+more'], {'id': 'related_entities[0]', 'type': "relationship_type})\n\n        # Add the relationship to the list of all relationships\n        all_relationships.append({\n            'type': relationship_type", 'start': "relationship['start']", 'end': "relationship['end']"}, {'title': "f'{", ".join(entity['id'] for entity in key_entities)}',\n        'summary'": '', 'rating': 0.0, 'rating_explanation': '', 'findings': []}, ["entity['id'"], {'f\'{entity["type': ''}, {'entity["id': ''}, ['entity[', 'description'], {'entity["id': ''}, {'entity["type': ''}, {'entity["description': ''}, {'entity["type': ''}, {'sum(entity["description': '.', 'report["findings': 'append(entity_info)\n                report[\'rating\'] += 0.2\n    if report[\'rating\'] > 10:\n        report[\'rating\'] = 10\n    report["findings"] += [\n    {\n        summary=f\'Re

Summarizing communities: 100%|██████████| 29/29 [45:42<00:00, 94.58s/it]


In [62]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License

"""DRIFT Search prompts."""

DRIFT_LOCAL_SYSTEM_PROMPT = """
---Role---

You are a helpful assistant responding to questions about data in the tables provided.


---Goal---

Generate a response of the target length and format that responds to the user's question, summarizing all information in the input data tables appropriate for the response length and format, and incorporating any relevant general knowledge.

If you don't know the answer, just say so. Do not make anything up.

Points supported by data should list their data references as follows:

"This is an example sentence supported by multiple data references [Data:  (record ids);  (record ids)]."

Do not list more than 5 record ids in a single reference. Instead, list the top 5 most relevant record ids and add "+more" to indicate that there are more.

For example:

"Person X is the owner of Company Y and subject to many allegations of wrongdoing [Data: Sources (15, 16)]."

where 15, 16, 1, 5, 7, 23, 2, 7, 34, 46, and 64 represent the id (not the index) of the relevant data record.

Pay close attention specifically to the Sources tables as it contains the most relevant information for the user query. You will be rewarded for preserving the context of the sources in your response.

---Target response length and format---

{response_type}


---Data tables---

{context_data}

---Goal---

Generate a response of the target length and format that responds to the user's question, summarizing all information in the input data tables appropriate for the response length and format, and incorporating any relevant general knowledge.

If you don't know the answer, just say so. Do not make anything up.

Points supported by data should list their data references as follows:

"This is an example sentence supported by multiple data references [Data:  (record ids);  (record ids)]."

Do not list more than 5 record ids in a single reference. Instead, list the top 5 most relevant record ids and add "+more" to indicate that there are more.

For example:

"Person X is the owner of Company Y and subject to many allegations of wrongdoing [Data: Sources (15, 16)]."

where 15, 16, 1, 5, 7, 23, 2, 7, 34, 46, and 64 represent the id (not the index) of the relevant data record.

Pay close attention specifically to the Sources tables as it contains the most relevant information for the user query. You will be rewarded for preserving the context of the sources in your response.

---Target response length and format---

{response_type}

Add sections and commentary to the response as appropriate for the length and format.

Additionally provide a score between 0 and 100 representing how well the response addresses the overall research question: {global_query}. Based on your response, suggest up to five follow-up questions that could be asked to further explore the topic as it relates to the overall research question. Do not include scores or follow up questions in the 'response' field of the JSON, add them to the respective 'score' and 'follow_up_queries' keys of the JSON output. Format your response in JSON with the following keys and values:

{{'response': str, Put your answer, formatted in markdown, here. Do not answer the global query in this section.
'score': int,
'follow_up_queries': List[str]}}
"""

DRIFT_REDUCE_PROMPT = """
---Role---

You are a helpful assistant responding to questions about data in the reports provided.

---Goal---

Generate a response of the target length and format that responds to the user's question, summarizing all information in the input reports appropriate for the response length and format, and incorporating any relevant general knowledge while being as specific, accurate and concise as possible.

If you don't know the answer, just say so. Do not make anything up.

Points supported by data should list their data references as follows:

"This is an example sentence supported by multiple data references [Data:  (record ids);  (record ids)]."

Do not list more than 5 record ids in a single reference. Instead, list the top 5 most relevant record ids and add "+more" to indicate that there are more.

For example:

"Person X is the owner of Company Y and subject to many allegations of wrongdoing [Data: Sources (1, 5, 15)]."

Do not include information where the supporting evidence for it is not provided.

If you decide to use general knowledge, you should add a delimiter stating that the information is not supported by the data tables. For example:

"Person X is the owner of Company Y and subject to many allegations of wrongdoing. [Data: General Knowledge (href)]"

---Data Reports---

{context_data}

---Target response length and format---

{response_type}

---Goal---

Generate a response of the target length and format that responds to the user's question, summarizing all information in the input reports appropriate for the response length and format, and incorporating any relevant general knowledge while being as specific, accurate and concise as possible.

If you don't know the answer, just say so. Do not make anything up.

Points supported by data should list their data references as follows:

"This is an example sentence supported by multiple data references [Data:  (record ids);  (record ids)]."

Do not list more than 5 record ids in a single reference. Instead, list the top 5 most relevant record ids and add "+more" to indicate that there are more.

For example:

"Person X is the owner of Company Y and subject to many allegations of wrongdoing [Data: Sources (1, 5, 15)]."

Do not include information where the supporting evidence for it is not provided.

If you decide to use general knowledge, you should add a delimiter stating that the information is not supported by the data tables. For example:

"Person X is the owner of Company Y and subject to many allegations of wrongdoing. [Data: General Knowledge (href)]".

Add sections and commentary to the response as appropriate for the length and format. Style the response in markdown. Now answer the following query using the data above:

"""


DRIFT_PRIMER_PROMPT = """You are a helpful agent designed to reason over a knowledge graph in response to a user query.
This is a unique knowledge graph where edges are freeform text rather than verb operators. You will begin your reasoning looking at a summary of the content of the most relevant communites and will provide:

1. score: How well the intermediate answer addresses the query. A score of 0 indicates a poor, unfocused answer, while a score of 100 indicates a highly focused, relevant answer that addresses the query in its entirety.

2. intermediate_answer: This answer should match the level of detail and length found in the community summaries. The intermediate answer should be exactly 2000 characters long. This must be formatted in markdown and must begin with a header that explains how the following text is related to the query.

3. follow_up_queries: A list of follow-up queries that could be asked to further explore the topic. These should be formatted as a list of strings. Generate at least five good follow-up queries.

Use this information to help you decide whether or not you need more information about the entities mentioned in the report. You may also use your general knowledge to think of entities which may help enrich your answer.

You will also provide a full answer from the content you have available. Use the data provided to generate follow-up queries to help refine your search. Do not ask compound questions, for example: "What is the market cap of Apple and Microsoft?". Use your knowledge of the entity distribution to focus on entity types that will be useful for searching a broad area of the knowledge graph.

For the query:

{query}

The top-ranked community summaries:

{community_reports}

Provide the intermediate answer, and all scores in JSON format following:

{{'intermediate_answer': str,
'score': int,
'follow_up_queries': List[str]}}

Begin:
"""

HYDE_PROMPT = """Create a hypothetical answer to the following query: {query}\n\n
          Format it to follow the structure of the template below:\n\n
          {template}\n"
          Ensure that the hypothetical answer does not reference new named entities that are not present in the original query."""

In [63]:
class CommunitySearch(Event):
    query: str
    hyde_query: str

class LocalSearch(Event):
    query: str
    local_query: str

class LocalSearchResults(Event):
    results: dict

class FinalAnswer(Event):
    query: str

In [ ]:
from my_graphrag.my_graphrag import async_retry
DEFAULT_RESPONSE_TYPE = "multiple paragraphs"
LOCAL_TOP_K = 3
MAX_LOCAL_SEARCH_DEPTH = 2
DRIFT_MODEL = "gpt-oss:20b"


@async_retry(max_retries=3, delay=1, backoff=2, exceptions=(Exception,))
async def generate_json(prompt, required_keys):
    # llama3.2 doesn't reliably follow the requested JSON schema, so retry on
    # malformed/non-dict output instead of letting the workflow crash.
    response = await client.chat.completions.create(
        model=DRIFT_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    parsed = json_repair.loads(response.choices[0].message.content)
    if not isinstance(parsed, dict) or not all(key in parsed for key in required_keys):
        raise ValueError(f"Model did not return the expected JSON object: {parsed!r}")
    return parsed


class DriftSearch(Workflow):
    @step
    async def hyde_generation(self, ev: StartEvent) -> CommunitySearch:
        # Fetch a random community report to use as a template for HyDE generation
        random_community_report = driver.execute_query(
            """
        MATCH (c:__Community__)
        WHERE c.summary IS NOT NULL
        RETURN coalesce(c.title, "") + " " + c.summary AS community_description""",
            result_transformer_=lambda r: r.data(),
        )
        # Generate a hypothetical answer to improve query representation
        hyde = HYDE_PROMPT.format(
            query=ev.query, template=random_community_report[0]["community_description"]
        )
        hyde_response = await client.chat.completions.create(
            model=DRIFT_MODEL,
            messages=[{"role": "user", "content": hyde}],
        )
        return CommunitySearch(query=ev.query, hyde_query=hyde_response.choices[0].message.content)

    @step
    async def community_search(self, ctx: Context, ev: CommunitySearch) -> LocalSearch:
        # Create embedding from the HyDE-enhanced query
        embedding_response = await client.embeddings.create(
            input=ev.hyde_query, model=TEXT_EMBEDDING_MODEL
        )
        embedding = embedding_response.data[0].embedding

        # Find top 5 most relevant community reports via vector similarity
        community_reports = driver.execute_query(
            """
        CALL db.index.vector.queryNodes('community', 5, $embedding) YIELD node, score
        RETURN 'community-' + node.id AS source_id, node.summary AS community_summary
        """,
            result_transformer_=lambda r: r.data(),
            embedding=embedding,
        )

        # Generate initial answer and identify what additional info is needed
        initial_prompt = DRIFT_PRIMER_PROMPT.format(
            query=ev.query, community_reports=community_reports
        )
        response_json = await generate_json(
            initial_prompt, ("intermediate_answer", "score", "follow_up_queries")
        )
        print(f"Initial intermediate response: {response_json['intermediate_answer']}")

        # Store the initial answer and prepare for parallel local searches
        async with ctx.store.edit_state() as ctx_state:
            ctx_state["intermediate_answers"] = [
                {
                    "intermediate_answer": response_json["intermediate_answer"],
                    "score": response_json["score"],
                }
            ]
            ctx_state["local_search_num"] = len(response_json["follow_up_queries"])

        # Dispatch follow-up queries to run in parallel
        for local_query in response_json["follow_up_queries"]:
            ctx.send_event(LocalSearch(query=ev.query, local_query=local_query))
        return None

    @step(num_workers=5)
    async def local_search(self, ev: LocalSearch) -> LocalSearchResults:
        print(f"Running local query: {ev.local_query}")

        # Create embedding for the local query
        response = await client.embeddings.create(
            input=ev.local_query, model=TEXT_EMBEDDING_MODEL
        )
        embedding = response.data[0].embedding

        # Retrieve relevant entities and gather their associated context:
        # - Text chunks where entities are mentioned
        # - Community reports the entities belong to
        # - Relationships between the retrieved entities
        # - Entity descriptions
        local_reports = driver.execute_query(
            """
    CALL db.index.vector.queryNodes('entity', 5, $embedding) YIELD node, score
    WITH collect(node) AS nodes
    WITH
    collect {
      UNWIND nodes as n
      MATCH (n)<-[:MENTIONS]->(c:__Chunk__)
      WITH c, count(distinct n) as freq
      RETURN {chunkText: c.text, source_id: 'chunk-' + c.id}
      ORDER BY freq DESC
      LIMIT 3
    } AS text_mapping,
    collect {
      UNWIND nodes as n
      MATCH (n)-[:IN_COMMUNITY*]->(c:__Community__)
      WHERE c.summary IS NOT NULL
      WITH c, c.rating as rank
      RETURN {summary: c.summary, source_id: 'community-' + c.id}
      ORDER BY rank DESC
      LIMIT 3
    } AS report_mapping,
    collect {
      UNWIND nodes as n
      MATCH (n)-[r:SUMMARIZED_RELATIONSHIP]-(m)
      WHERE m IN nodes
      RETURN {descriptionText: r.summary, source_id: 'relationship-' + n.name + '-' + m.name}
      LIMIT 3
    } as insideRels,
    collect {
      UNWIND nodes as n
      RETURN {descriptionText: n.summary, source_id: 'node-' + n.name}
    } as entities
    RETURN {Chunks: text_mapping, Reports: report_mapping,
      Relationships: insideRels,
      Entities: entities} AS output
    """,
            result_transformer_=lambda r: r.data(),
            embedding=embedding,
        )

        # Generate answer based on the retrieved context
        local_prompt = DRIFT_LOCAL_SYSTEM_PROMPT.format(
            response_type=DEFAULT_RESPONSE_TYPE,
            context_data=local_reports,
            global_query=ev.query,
        )
        response_json = await generate_json(
            local_prompt, ("response", "score", "follow_up_queries")
        )

        # Limit follow-up queries to prevent exponential growth
        response_json["follow_up_queries"] = response_json["follow_up_queries"][:LOCAL_TOP_K]

        return LocalSearchResults(results=response_json, query=ev.query)
    @step
    async def local_search_results(
        self, ctx: Context, ev: LocalSearchResults
    ) -> LocalSearch | FinalAnswer:
        local_search_num = await ctx.store.get("local_search_num")

        # Wait for all parallel searches to complete
        results = ctx.collect_events(ev, [LocalSearchResults] * local_search_num)
        if results is None:
            return None

        intermediate_results = [
            {
                "intermediate_answer": event.results["response"],
                "score": event.results["score"],
            }
            for event in results
        ]
        current_depth = await ctx.store.get("local_search_depth", default=1)
        query = [ev.query for ev in results][0]

        # Continue drilling down if we haven't reached max depth
        if current_depth < MAX_LOCAL_SEARCH_DEPTH:
            await ctx.store.set("local_search_depth", current_depth + 1)
            follow_up_queries = [
                query
                for event in results
                for query in event.results["follow_up_queries"]
            ]

            # Store intermediate answers and dispatch next round of searches
            async with ctx.store.edit_state() as ctx_state:
                ctx_state["intermediate_answers"].extend(intermediate_results)
                ctx_state["local_search_num"] = len(follow_up_queries)

            for local_query in follow_up_queries:
                ctx.send_event(LocalSearch(query=query, local_query=local_query))
            return None
        else:
            return FinalAnswer(query=query)
    @step
    async def final_answer_generation(self, ctx: Context, ev: FinalAnswer) -> StopEvent:
        # Retrieve all intermediate answers collected throughout the search process
        intermediate_answers = await ctx.store.get("intermediate_answers")

        # Synthesize all findings into a comprehensive final response
        answer_prompt = DRIFT_REDUCE_PROMPT.format(
            response_type=DEFAULT_RESPONSE_TYPE,
            context_data=intermediate_answers,
            global_query=ev.query,
        )
        answer_response = await client.chat.completions.create(
            model=DRIFT_MODEL,
            messages=[
                {"role": "system", "content": answer_prompt},
                {"role": "user", "content": ev.query},
            ],
        )

        return StopEvent(result=answer_response.choices[0].message.content)

In [65]:
global_question = """How do the various characters Alice encounters in Wonderland, such as the Cheshire Cat, the Caterpillar,
and the Queen of Hearts, challenge her understanding of logic, rules, and authority, and what does this reveal about the nature of the adult world she is entering?"""

w = DriftSearch(timeout=36000, verbose=False)
result = await w.run(query=global_question)

Initial intermediate response: # The Tale of Logic, Rules, and Authority in Wonderland

Alice’s journey into the chaotic realm of Wonderland is more than a whimsical escape; it is a series of encounters that systematically destabilize her assumptions about reason, law, and power.  Each character she meets operates on a logic that contradicts ordinary experience, forcing her to confront the instability of the adult world she is stepping toward.

**Cheshire Cat** – the Cat’s grin appears and disappears at will, and it keeps telling Alice that “we are all mad here.”  His conversational jests and cryptic wisdom break the binary of “right or wrong” and show that identity can be fluid, that language is playful, and that authority is often a matter of presence rather than command.  Alice learns that following a simple rule (“stay in bed for nine days, my dear”) is not enough to capture the whole system; logic can be a suggestion, not a law.

**Caterpillar** – perched on a mushroom and smoking

In [66]:
print(result)

## How Wonderland’s Characters Shake Alice’s Reasoning

Alice’s journey through Wonderland is punctuated by symbolic encounters that systematically dismantle her child‑like intuitions about logic, rules, and authority. The main figures – the Cheshire Cat, the Caterpillar, and the Queen of Hearts – each test a different aspect of her worldview, and their interactions together reveal a broader critique of adult social structures.

### 1.  The Cheshire Cat: “Identity is Inconstant”

The Cat’s ability to appear, vanish, and return with only a grin challenges the idea that an object’s identity is fixed. Alice is forced to confront that “what you see” can be an unreliable basis for logical deduction.  
- *Data:* The cat’s disappearing grin is documented in the text fragment describing the Cat’s grin “appears and disappears at will” and the quote “we are all mad here” [Data: chunk-c8612f9b458d0873cf8ace73aa780103; node-CHESHIRE CAT +more].

### 2.  The Caterpillar: “Perspective Alters Reality